# Epstein Files – Exploratory Data Analysis

**Dataset:** [`notesbymuneeb/epstein-emails`](https://huggingface.co/datasets/notesbymuneeb/epstein-emails)  
**Purpose:** Understand the raw email corpus before classification.

---
### Notebook outline
1. Environment setup
2. Dataset loading
3. Schema & basic statistics
4. Null / missing value audit
5. Message parsing
6. Text-length distributions
7. Message-count per thread
8. Sender analysis
9. Subject-line analysis
10. Date / timeline analysis
11. Key takeaways

## 1 · Environment Setup

In [ ]:
# Install dependencies (run once; comment out if already installed)
!pip install -q datasets pandas numpy plotly tqdm wordcloud matplotlib

In [ ]:
import json
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datasets import load_dataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
tqdm.pandas()

print('✅ Imports OK')

## 2 · Dataset Loading

In [ ]:
print('Downloading dataset …')
raw = load_dataset('notesbymuneeb/epstein-emails')
print(raw)

df_raw = raw['train'].to_pandas()
print(f'\nShape: {df_raw.shape}')
df_raw.head(3)

## 3 · Schema & Basic Statistics

In [ ]:
print('=== Column types ===')
print(df_raw.dtypes)
print('\n=== Memory usage ===')
print(df_raw.memory_usage(deep=True).sum() / 1024**2, 'MB')

## 4 · Null / Missing Value Audit

In [ ]:
null_counts = df_raw.isnull().sum()
null_pct    = (null_counts / len(df_raw) * 100).round(2)

null_df = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
null_df = null_df[null_df['null_count'] > 0].sort_values('null_count', ascending=False)
print(null_df)

fig = px.bar(
    null_df.reset_index(),
    x='index', y='null_pct',
    title='Missing Values by Column (%)',
    labels={'index': 'Column', 'null_pct': 'Missing (%)'},
    color_discrete_sequence=['#C0392B'],
    text='null_pct',
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(template='plotly_white')
fig.show()

## 5 · Message Parsing

Each row's `messages` field contains a JSON list of individual message dicts.  
We parse them into flat columns for downstream analysis.

In [ ]:
def parse_messages(field):
    """Return (full_text, senders, date_range, msg_count)."""
    try:
        msgs = json.loads(field) if isinstance(field, str) else field
        if not isinstance(msgs, list):
            return '', [], 'Unknown', 0

        bodies, senders, dates = [], [], []
        for m in msgs:
            b = m.get('body', '') or m.get('content', '') or ''
            s = m.get('sender', '') or m.get('from', '') or ''
            d = m.get('date', '') or m.get('timestamp', '') or ''
            if b: bodies.append(str(b).strip())
            if s: senders.append(str(s).strip())
            if d: dates.append(str(d).strip())

        full_text  = ' [MSG] '.join(bodies)
        uniq_s     = list(dict.fromkeys(senders))
        date_range = (
            f"{dates[0]} → {dates[-1]}" if len(dates) >= 2
            else (dates[0] if dates else 'Unknown')
        )
        return full_text, uniq_s, date_range, len(msgs)
    except Exception:
        return '', [], 'Unknown', 0


def clean_text(text):
    if not text or not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'(From|To|Subject|Date|Cc|Bcc):\s*[^\n]+\n', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\x20-\x7E]', ' ', text)
    return text.strip()


print('Parsing messages …')
parsed = df_raw['messages'].progress_apply(parse_messages)

df = df_raw.copy()
df['full_text']     = [clean_text(p[0]) for p in parsed]
df['senders']       = [p[1] for p in parsed]
df['date_range']    = [p[2] for p in parsed]
df['message_count'] = [p[3] for p in parsed]
df['subject']       = df['subject'].fillna('[No Subject]')
df['text_len']      = df['full_text'].str.len()

# Drop empty threads
before = len(df)
df = df[df['text_len'] > 20].reset_index(drop=True)
print(f'Removed {before - len(df)} empty threads → {len(df):,} remaining')
df[['subject', 'date_range', 'message_count', 'text_len']].head()

## 6 · Text-Length Distribution

In [ ]:
print(df['text_len'].describe().round(1))

fig = px.histogram(
    df, x='text_len', nbins=60,
    title='Distribution of Thread Text Length (chars)',
    labels={'text_len': 'Character count'},
    color_discrete_sequence=['#2E5FA3'],
)
fig.update_layout(template='plotly_white')
fig.show()

# Log-scale view for long-tail
fig2 = px.histogram(
    df, x='text_len', nbins=60, log_y=True,
    title='Text Length (log scale)',
    color_discrete_sequence=['#2E5FA3'],
)
fig2.update_layout(template='plotly_white')
fig2.show()

## 7 · Messages per Thread

In [ ]:
print(df['message_count'].describe().round(1))

# Cap display at 20 messages for readability
capped = df['message_count'].clip(upper=20)
fig = px.histogram(
    capped, nbins=20,
    title='Messages per Thread (capped at 20)',
    labels={'value': 'Messages in thread'},
    color_discrete_sequence=['#2E5FA3'],
)
fig.update_layout(template='plotly_white')
fig.show()

## 8 · Sender Analysis

In [ ]:
all_senders = [s for slist in df['senders'] for s in slist]
top_senders = pd.Series(all_senders).value_counts().head(25)

print(f'Unique senders: {len(set(all_senders)):,}')
print(f'Total sender mentions: {len(all_senders):,}')

fig = px.bar(
    x=top_senders.values,
    y=top_senders.index,
    orientation='h',
    title='Top 25 Senders (by thread appearances)',
    labels={'x': 'Appearances', 'y': 'Sender'},
    color_discrete_sequence=['#C0392B'],
    text=top_senders.values,
)
fig.update_traces(textposition='outside')
fig.update_layout(template='plotly_white', height=650)
fig.show()

In [ ]:
# Threads per unique sender count
df['n_senders'] = df['senders'].apply(len)

fig = px.histogram(
    df, x='n_senders', nbins=20,
    title='Number of Unique Senders per Thread',
    labels={'n_senders': 'Unique senders'},
    color_discrete_sequence=['#7F8C8D'],
)
fig.update_layout(template='plotly_white')
fig.show()

## 9 · Subject-Line Analysis

In [ ]:
# Most common words in subjects
STOP_WORDS = {
    'the','a','an','and','or','but','in','on','at','to','for',
    'of','with','is','it','re','fw','fwd','no subject','subject',''
}

subject_words = [
    w.lower().strip('.,!?;:"\'')
    for subj in df['subject']
    for w in str(subj).split()
    if w.lower().strip('.,!?;:"\'') not in STOP_WORDS
]

top_words = pd.Series(subject_words).value_counts().head(30)

fig = px.bar(
    x=top_words.index, y=top_words.values,
    title='Top 30 Words in Email Subjects',
    labels={'x': 'Word', 'y': 'Frequency'},
    color_discrete_sequence=['#2E5FA3'],
)
fig.update_layout(template='plotly_white', xaxis_tickangle=-45)
fig.show()

In [ ]:
# Word cloud (optional – requires wordcloud package)
try:
    from wordcloud import WordCloud
    import matplotlib.pyplot as plt

    text_blob = ' '.join(
        w for w in subject_words if len(w) > 2
    )
    wc = WordCloud(
        width=900, height=400, background_color='white',
        colormap='Reds', max_words=100,
    ).generate(text_blob)

    plt.figure(figsize=(12, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title('Subject Word Cloud', fontsize=14)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('wordcloud not installed – skipping. Run: pip install wordcloud')

## 10 · Date / Timeline Analysis

In [ ]:
# Extract the start date from date_range strings
def extract_start_year(date_range_str):
    if not isinstance(date_range_str, str):
        return None
    # Look for a 4-digit year
    match = re.search(r'(19|20)\d{2}', date_range_str)
    return int(match.group()) if match else None

df['year'] = df['date_range'].apply(extract_start_year)

year_counts = df['year'].dropna().astype(int).value_counts().sort_index()
print(f'Year range: {int(year_counts.index.min())} – {int(year_counts.index.max())}')

fig = px.bar(
    x=year_counts.index.astype(str), y=year_counts.values,
    title='Email Threads by Year',
    labels={'x': 'Year', 'y': 'Thread count'},
    color_discrete_sequence=['#2E5FA3'],
    text=year_counts.values,
)
fig.update_traces(textposition='outside')
fig.update_layout(template='plotly_white')
fig.show()

## 11 · Key Takeaways

| Metric | Value |
|--------|-------|
| Total threads | — |
| Threads with text > 20 chars | — |
| Median text length (chars) | — |
| Unique senders | — |
| Date range | — |

_Fill in values after running the cells above._

### Observations
* **Subject lines** are sparse – many threads have `[No Subject]`, reinforcing the need for body-text analysis.
* **Text lengths** are highly skewed – a small number of threads contain very long message chains.
* **Senders** follow a power-law distribution; a handful of addresses appear across many threads.
* **Temporal coverage** spans roughly the 1990s–2010s, aligning with the Epstein investigation timeline.

→ These observations inform the **DEMO_MODE** default of 200 threads and the **512-character truncation** used in classification.

---
_Proceed to `python main.py` to run the full classification pipeline._